# Week 7 — Advanced models with LightGBM

This notebook tunes a gradient-boosted tree regressor for California
single-family close prices. Run it from the repository root after
installing requirements-week7.txt. It reads the same
csv/cleaned_sales.csv export used in the earlier notebooks.

The export has 29 base numeric predictors and 12 target-encoded
categorical predictors. Those encodings were cross-fitted across the
entire April 2025–March 2026 development period. To avoid label
leakage into a new validation split, this experiment uses only the
29 base predictors. Exported missingness flags restore missing values;
LightGBM handles them directly.

Hyperparameters are selected on January–March 2026 validation sales.
The selected model is refit on all development sales, then scored on
April 2026. That test month appeared in earlier notebooks, so these
results are a reused holdout comparison.


In [1]:
from pathlib import Path
from time import perf_counter
import platform

import lightgbm
import numpy as np
import pandas as pd
import sklearn
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DATA_PATH = Path("csv/cleaned_sales.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "Run 02_preprocessing.ipynb from the repository root to create "
        "csv/cleaned_sales.csv."
    )
print(
    "Python", platform.python_version(),
    "| NumPy", np.__version__,
    "| pandas", pd.__version__,
    "| scikit-learn", sklearn.__version__,
    "| LightGBM", lightgbm.__version__,
)


Python 3.12.14 | NumPy 2.3.5 | pandas 3.0.1 | scikit-learn 1.7.2 | LightGBM 4.6.0


## 1. Chronological validation split

ClosePrice is the label; CloseDate and DataSplit are audit fields.
The model receives neither these fields nor the 12 target encodings.
Missing-indicator flags undo the earlier median fills before fitting.


In [2]:
data = pd.read_csv(DATA_PATH, parse_dates=["CloseDate"])
metadata_columns = ["CloseDate", "ClosePrice", "DataSplit"]
base_columns = [
    "LivingArea", "BelowGradeFinishedArea", "BedroomsTotal",
    "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
    "PropertyAge", "Latitude", "Longitude", "AssociationFee",
    "GarageSpaces", "ParkingTotal", "Stories", "MainLevelBedrooms",
    "AsOfYear", "AsOfMonthSin", "AsOfMonthCos",
    "missingindicator_BelowGradeFinishedArea",
    "missingindicator_BathroomsTotalInteger",
    "missingindicator_LotSizeSquareFeet",
    "missingindicator_YearBuilt", "missingindicator_PropertyAge",
    "missingindicator_Latitude", "missingindicator_Longitude",
    "missingindicator_AssociationFee",
    "missingindicator_GarageSpaces",
    "missingindicator_ParkingTotal",
    "missingindicator_Stories",
    "missingindicator_MainLevelBedrooms",
]
encoded_columns = [
    "ViewYN", "WaterfrontYN", "BasementYN", "PoolPrivateYN",
    "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "CountyOrParish", "City", "PostalCode",
    "HighSchoolDistrict", "Levels",
]
if list(data.columns) != metadata_columns + base_columns + encoded_columns:
    raise ValueError(
        "Unexpected export schema. Rebuild csv/cleaned_sales.csv "
        "with 02_preprocessing.ipynb."
    )
if set(data["DataSplit"]) != {"train", "test"}:
    raise ValueError("DataSplit must contain exactly train and test.")
if data["CloseDate"].isna().any():
    raise ValueError("CloseDate cannot be missing.")

X = data.loc[:, base_columns].copy()
y = pd.to_numeric(data["ClosePrice"], errors="raise")
if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Exported base predictors must be finite.")
if not np.isfinite(y.to_numpy(dtype=float)).all() or not y.gt(0).all():
    raise ValueError("ClosePrice must be finite and positive.")
restored_missing = 0
for flag in (c for c in base_columns if c.startswith("missingindicator_")):
    source = flag.removeprefix("missingindicator_")
    if source not in X or not X[flag].isin([0, 1]).all():
        raise ValueError(f"Invalid missingness flag: {flag}")
    missing = X[flag].eq(1)
    restored_missing += int(missing.sum())
    X[source] = X[source].astype(float).mask(missing)

development_mask = data["DataSplit"].eq("train")
test_mask = data["DataSplit"].eq("test")
validation_start = pd.Timestamp("2026-01-01")
fit_mask = development_mask & data["CloseDate"].lt(validation_start)
validation_mask = development_mask & data["CloseDate"].ge(validation_start)
if not all(m.any() for m in
           [fit_mask, validation_mask, development_mask, test_mask]):
    raise ValueError("A chronological partition is empty.")
if not (
    data.loc[fit_mask, "CloseDate"].max()
    < data.loc[validation_mask, "CloseDate"].min()
    and data.loc[development_mask, "CloseDate"].max()
    < data.loc[test_mask, "CloseDate"].min()
):
    raise ValueError("Split dates overlap or are out of order.")

split_rows = []
for name, mask in [
    ("Tuning fit", fit_mask), ("Validation", validation_mask),
    ("All development", development_mask), ("Reused test", test_mask),
]:
    dates = data.loc[mask, "CloseDate"]
    split_rows.append({
        "Partition": name, "Rows": int(mask.sum()),
        "First close": dates.min().date().isoformat(),
        "Last close": dates.max().date().isoformat(),
    })
print(f"Base numeric predictors: {len(base_columns)}")
print(f"Restored missing numeric values: {restored_missing:,}")
print(pd.DataFrame(split_rows).to_string(index=False))


Base numeric predictors: 29
Restored missing numeric values: 262,456
      Partition   Rows First close Last close
     Tuning fit 102321  2025-04-01 2025-12-31
     Validation  27953  2026-01-01 2026-03-31
All development 130274  2025-04-01 2026-03-31
    Reused test  12470  2026-04-01 2026-04-30


## 2. Light hyperparameter tuning

Four prespecified candidates vary maximum depth, learning rate, and
number of boosting trees (n_estimators). Validation MAE in dollars
selects the configuration; RMSE breaks a tie. April labels are unused.


In [3]:
candidates = [
    {"max_depth": 4, "learning_rate": 0.08, "n_estimators": 250},
    {"max_depth": 6, "learning_rate": 0.05, "n_estimators": 400},
    {"max_depth": 8, "learning_rate": 0.05, "n_estimators": 500},
    {"max_depth": 6, "learning_rate": 0.10, "n_estimators": 250},
]

def make_model(parameters):
    depth = parameters["max_depth"]
    return LGBMRegressor(
        objective="regression",
        max_depth=depth,
        num_leaves=min(2 ** depth - 1, 63),
        learning_rate=parameters["learning_rate"],
        n_estimators=parameters["n_estimators"],
        min_child_samples=40,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=4,
        deterministic=True,
        force_col_wise=True,
        verbosity=-1,
    )

tuning_rows = []
for number, parameters in enumerate(candidates, start=1):
    model = make_model(parameters)
    started = perf_counter()
    model.fit(X.loc[fit_mask], y.loc[fit_mask])
    fit_seconds = perf_counter() - started
    predicted = model.predict(X.loc[validation_mask])
    actual = y.loc[validation_mask]
    tuning_rows.append({
        "Candidate": number,
        "max_depth": parameters["max_depth"],
        "learning_rate": parameters["learning_rate"],
        "n_estimators": parameters["n_estimators"],
        "Validation MAE ($)": mean_absolute_error(actual, predicted),
        "Validation RMSE ($)": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        "Validation R²": r2_score(actual, predicted),
        "Fit seconds": fit_seconds,
    })
    print(f"Fitted candidate {number} in {fit_seconds:.1f} s")

tuning = pd.DataFrame(tuning_rows).sort_values(
    ["Validation MAE ($)", "Validation RMSE ($)", "Candidate"]
).reset_index(drop=True)
selected_number = int(tuning.loc[0, "Candidate"])
selected_parameters = candidates[selected_number - 1]
print("\nValidation results, ranked by MAE:")
print(tuning.round({
    "learning_rate": 3, "Validation MAE ($)": 0,
    "Validation RMSE ($)": 0, "Validation R²": 4,
    "Fit seconds": 1,
}).to_string(index=False))
print(f"Selected candidate {selected_number}: {selected_parameters}")


Fitted candidate 1 in 0.6 s
Fitted candidate 2 in 1.2 s
Fitted candidate 3 in 1.5 s
Fitted candidate 4 in 0.7 s

Validation results, ranked by MAE:
 Candidate  max_depth  learning_rate  n_estimators  Validation MAE ($)  Validation RMSE ($)  Validation R²  Fit seconds
         3          8           0.05           500            216474.0             558862.0         0.8248          1.5
         4          6           0.10           250            227021.0             567180.0         0.8195          0.7
         2          6           0.05           400            230680.0             571560.0         0.8167          1.2
         1          4           0.08           250            256450.0             602659.0         0.7962          0.6
Selected candidate 3: {'max_depth': 8, 'learning_rate': 0.05, 'n_estimators': 500}


## 3. Refit and report test metrics

Refit the selected configuration on every April 2025–March 2026
development sale. Score the April 2026 test rows once. Negative
dollar estimates, if any, are clipped to zero for reported metrics.


In [4]:
final_model = make_model(selected_parameters)
started = perf_counter()
final_model.fit(X.loc[development_mask], y.loc[development_mask])
final_fit_seconds = perf_counter() - started
raw_predictions = final_model.predict(X.loc[test_mask])
predictions = np.maximum(raw_predictions, 0.0)
actual = y.loc[test_mask].to_numpy(dtype=float)
ape = np.abs(predictions - actual) / actual
test_metrics = {
    "rows": len(actual),
    "r2": r2_score(actual, predictions),
    "mae": mean_absolute_error(actual, predictions),
    "rmse": np.sqrt(mean_squared_error(actual, predictions)),
    "median_ape": np.median(ape),
    "within_10pct": np.mean(ape <= 0.10),
    "negative_raw_predictions": int(np.sum(raw_predictions < 0)),
}
print(f"Selected hyperparameters: {selected_parameters}")
print(f"Test rows: {test_metrics['rows']:,}")
print(f"Test R²: {test_metrics['r2']:.4f}")
print("Test MAE: $" + f"{test_metrics['mae']:,.0f}")
print("Test RMSE: $" + f"{test_metrics['rmse']:,.0f}")
print(f"Test median absolute percentage error: {test_metrics['median_ape']:.2%}")
print(f"Test predictions within 10%: {test_metrics['within_10pct']:.2%}")
print(f"Negative raw test predictions: {test_metrics['negative_raw_predictions']}")
print(f"Final fit time: {final_fit_seconds:.1f} s")


Selected hyperparameters: {'max_depth': 8, 'learning_rate': 0.05, 'n_estimators': 500}
Test rows: 12,470
Test R²: 0.8348
Test MAE: $219,861
Test RMSE: $551,056
Test median absolute percentage error: 10.74%
Test predictions within 10%: 47.27%
Negative raw test predictions: 1
Final fit time: 1.9 s


## 4. Interpretation and limits

This evaluates closed MLS sales in the reused April 2026 holdout.
The notebook excludes the target-encoded location and category
features to keep its internal tuning split free of label leakage.
Its scores are not an isolated algorithm comparison against Weeks
5–6, which used 41 or 44 predictors. A later, untouched sales
period is needed for independent confirmation.
